In [1]:
# Практика 15. Критерії узгодженості й таблиці спряженості
# Коць Артем, ІТ-42
# Варіант 9 — Суми

import numpy as np
import pandas as pd
from scipy import stats

np.random.seed(42)

variant = 9
city = "Суми"
region = "Схід"
base_temp = 7.5
amplitude = 14

def season(month):
    if month in (12, 1, 2):
        return "зима"
    if month in (3, 4, 5):
        return "весна"
    if month in (6, 7, 8):
        return "літо"
    return "осінь"

rows = []

for year in [2021, 2022, 2023, 2024]:
    for month in range(1, 13):
        seasonal = amplitude * np.cos(
            (month - 7) / 12 * 2 * np.pi
        )
        
        noise = np.random.normal(0, 1.0)
        temp = round(base_temp + seasonal + noise, 1)
        
        diff = temp - base_temp
        
        if diff < -3:
            norm_cat = "холодніше"
        elif diff > 3:
            norm_cat = "тепліше"
        else:
            norm_cat = "звичайно"
        
        rows.append({
            "місто": city,
            "рік": year,
            "місяць": month,
            "температура": temp,
            "сезон": season(month),
            "відхилення_від_норми": norm_cat
        })

climate = pd.DataFrame(rows)

print("Місто:", city)
print("Варіант:", variant)
print("Регіон:", region)
print("Середньорічна температура:", base_temp, "°C")
print("Сезонна амплітуда:", amplitude, "°C")
print("Кількість рядків:", len(climate))

display(climate.head(12))

Місто: Суми
Варіант: 9
Регіон: Схід
Середньорічна температура: 7.5 °C
Сезонна амплітуда: 14 °C
Кількість рядків: 48


,місто,рік,місяць,температура,сезон,відхилення_від_норми
0,Суми,2021,1,-6.0,зима,холодніше
1,Суми,2021,2,-4.8,зима,холодніше
2,Суми,2021,3,1.1,весна,холодніше
3,Суми,2021,4,9.0,весна,звичайно
4,Суми,2021,5,14.3,весна,тепліше
5,Суми,2021,6,19.4,літо,тепліше
6,Суми,2021,7,23.1,літо,тепліше
7,Суми,2021,8,20.4,літо,тепліше
8,Суми,2021,9,14.0,осінь,тепліше
9,Суми,2021,10,8.0,осінь,звичайно


In [2]:
print("Розмір DataFrame:", climate.shape)

print("\nСтовпці:")
print(climate.columns.tolist())

print("\nСезони:")
print(climate["сезон"].value_counts())

print("\nВідхилення від норми:")
print(climate["відхилення_від_норми"].value_counts())

Розмір DataFrame: (48, 6)

Стовпці:
['місто', 'рік', 'місяць', 'температура', 'сезон', 'відхилення_від_норми']

Сезони:
сезон
зима     12
весна    12
літо     12
осінь    12
Name: count, dtype: int64

Відхилення від норми:
відхилення_від_норми
холодніше    20
тепліше      20
звичайно      8
Name: count, dtype: int64


In [3]:
# Завдання 1. Таблиця спряженості

table = pd.crosstab(
    climate["сезон"],
    climate["відхилення_від_норми"]
)

print("Таблиця спряженості:")
display(table)

Таблиця спряженості:


відхилення_від_норми,звичайно,тепліше,холодніше
сезон,,,
весна,4,4,4
зима,0,0,12
літо,0,12,0
осінь,4,4,4


In [4]:
table_normalized = pd.crosstab(
    climate["сезон"],
    climate["відхилення_від_норми"],
    normalize="index"
)

print("Нормована таблиця спряженості:")
display(table_normalized.round(3))

print("\nСума часток у кожному рядку:")
print(table_normalized.sum(axis=1))

Нормована таблиця спряженості:


відхилення_від_норми,звичайно,тепліше,холодніше
сезон,,,
весна,0.333,0.333,0.333
зима,0.000,0.000,1.000
літо,0.000,1.000,0.000
осінь,0.333,0.333,0.333



Сума часток у кожному рядку:
сезон
весна    1.0
зима     1.0
літо     1.0
осінь    1.0
dtype: float64


### Завдання 1. Таблиця спряженості

За допомогою pd.crosstab() побудовано таблицю спряженості
для категоріальних змінних "сезон" і "відхилення_від_норми".

Сира таблиця показує кількість спостережень кожної комбінації
сезону та категорії відхилення.

Найчастіші комбінації можна визначити безпосередньо за найбільшими
значеннями таблиці. Отриманий результат узгоджується з побудовою
набору, оскільки сезонна температура формується навколо
середньорічної температури 7.5 °C.

Нормована таблиця з normalize="index" показує частки категорій
всередині кожного сезону замість абсолютних кількостей.
Тому сума значень кожного рядка дорівнює 1.

Нормування зручне для порівняння сезонів, оскільки дозволяє
порівнювати структуру категорій незалежно від кількості
спостережень у кожному рядку.

In [5]:
# Завдання 2. Критерій незалежності хі-квадрат

chi2, p_value, dof, expected = stats.chi2_contingency(table)

expected_table = pd.DataFrame(
    expected,
    index=table.index,
    columns=table.columns
)

print("χ² =", chi2)
print("p-value =", p_value)
print("Ступені свободи =", dof)

print("\nОчікувані частоти:")
display(expected_table.round(2))

print("\nНайбільша абсолютна різниця |observed - expected|:")

difference = (table - expected_table).abs()

max_cell = difference.stack().idxmax()
max_difference = difference.stack().max()

print("Клітинка:", max_cell)
print("Різниця:", max_difference)
print("Observed =", table.loc[max_cell])
print("Expected =", expected_table.loc[max_cell])

χ² = 38.400000000000006
p-value = 9.381704108243484e-07
Ступені свободи = 6

Очікувані частоти:


відхилення_від_норми,звичайно,тепліше,холодніше
сезон,,,
весна,2.0,5.0,5.0
зима,2.0,5.0,5.0
літо,2.0,5.0,5.0
осінь,2.0,5.0,5.0



Найбільша абсолютна різниця |observed - expected|:
Клітинка: ('зима', 'холодніше')
Різниця: 7.0
Observed = 12
Expected = 5.0


### Висновок до завдання 2

Для таблиці спряженості отримано:

- χ² = 38.4
- p-value ≈ 0.000001
- ступені свободи = 6
- рівень значущості α = 0.05.

Оскільки p-value < 0.05, нульову гіпотезу про незалежність сезону та категорії відхилення температури від норми відхиляємо.

Отже, між сезоном і категорією відхилення температури від норми існує статистично значущий зв'язок.

Найбільша різниця між спостережуваною та очікуваною частотами виникає для клітинок, де спостережуване значення 0, а очікуване значення дорівнює 5.

In [6]:
# Перевірка очікуваних частот

print("Очікувані частоти:")
display(expected_table.round(2))

print("\nМінімальна очікувана частота:", expected_table.min().min())

if (expected_table >= 5).all().all():
    print("Усі очікувані частоти >= 5.")
else:
    print("Є очікувані частоти < 5.")

Очікувані частоти:


відхилення_від_норми,звичайно,тепліше,холодніше
сезон,,,
весна,2.0,5.0,5.0
зима,2.0,5.0,5.0
літо,2.0,5.0,5.0
осінь,2.0,5.0,5.0



Мінімальна очікувана частота: 2.0
Є очікувані частоти < 5.


In [7]:
# Об'єднання категорій для виконання умови очікуваних частот

climate["відхилення_об'єднане"] = climate["відхилення_від_норми"].replace({
    "звичайно": "не холодніше",
    "тепліше": "не холодніше"
})

table_combined = pd.crosstab(
    climate["сезон"],
    climate["відхилення_об'єднане"]
)

print("Об'єднана таблиця спряженості:")
display(table_combined)

Об'єднана таблиця спряженості:


відхилення_об'єднане,не холодніше,холодніше
сезон,,
весна,8,4
зима,0,12
літо,12,0
осінь,8,4


In [8]:
# Повторний критерій хі-квадрат

chi2_combined, p_combined, dof_combined, expected_combined = stats.chi2_contingency(
    table_combined
)

expected_combined_table = pd.DataFrame(
    expected_combined,
    index=table_combined.index,
    columns=table_combined.columns
)

print("χ² =", chi2_combined)
print("p-value =", p_combined)
print("Ступені свободи =", dof_combined)

print("\nОчікувані частоти після об'єднання:")
display(expected_combined_table.round(2))

print("\nМінімальна очікувана частота:",
      expected_combined_table.min().min())

χ² = 26.057142857142857
p-value = 9.278240763009508e-06
Ступені свободи = 3

Очікувані частоти після об'єднання:


відхилення_об'єднане,не холодніше,холодніше
сезон,,
весна,7.0,5.0
зима,7.0,5.0
літо,7.0,5.0
осінь,7.0,5.0



Мінімальна очікувана частота: 5.0


In [9]:
# Завдання 4. Перевірка рівномірності сезонів

season_counts = climate["сезон"].value_counts().reindex(
    ["зима", "весна", "літо", "осінь"]
)

print("Кількість спостережень за сезонами:")
display(season_counts)

print("\nЧастки:")
display((season_counts / len(climate)).round(3))

Кількість спостережень за сезонами:


сезон
зима     12
весна    12
літо     12
осінь    12
Name: count, dtype: int64


Частки:


сезон
зима     0.25
весна    0.25
літо     0.25
осінь    0.25
Name: count, dtype: float64

In [10]:
# Очікуємо однакову кількість спостережень у кожному сезоні

expected_seasons = [len(climate) / 4] * 4

chi2_seasons, p_seasons = stats.chisquare(
    f_obs=season_counts.values,
    f_exp=expected_seasons
)

print("Спостережувані частоти:", season_counts.values)
print("Очікувані частоти:", expected_seasons)
print("χ² =", chi2_seasons)
print("p-value =", p_seasons)

alpha = 0.05

if p_seasons < alpha:
    print("Висновок: відхиляємо H0.")
else:
    print("Висновок: немає підстав відхиляти H0.")

Спостережувані частоти: [12 12 12 12]
Очікувані частоти: [12.0, 12.0, 12.0, 12.0]
χ² = 0.0
p-value = 1.0
Висновок: немає підстав відхиляти H0.


### Висновок до завдання 4

Нульова гіпотеза H₀: чотири сезони представлені однаково, тобто кожен сезон має теоретичну частку 25%.

Для кожного сезону отримано по 12 спостережень із 48 загальних, тобто по 25%.

Критерій χ² дав значення χ² = 0 та p-value = 1.0.

Оскільки p-value > 0.05, немає підстав відхиляти нульову гіпотезу.

Отже, розподіл спостережень за сезонами відповідає рівномірному розподілу 25% / 25% / 25% / 25%.

In [11]:
# Завдання 5. Перевірка власної гіпотези про пропорції

deviation_counts = climate["відхилення_від_норми"].value_counts().reindex(
    ["холодніше", "звичайно", "тепліше"],
    fill_value=0
)

print("Фактичні частоти:")
display(deviation_counts)

proportions = np.array([0.25, 0.25, 0.50])

expected_deviation = proportions * len(climate)

chi2_deviation, p_deviation = stats.chisquare(
    f_obs=deviation_counts.values,
    f_exp=expected_deviation
)

print("Очікувані частоти:")
print(expected_deviation)

print("\nχ² =", chi2_deviation)
print("p-value =", p_deviation)

Фактичні частоти:


відхилення_від_норми
холодніше    20
звичайно      8
тепліше      20
Name: count, dtype: int64

Очікувані частоти:
[12. 12. 24.]

χ² = 7.333333333333333
p-value = 0.025561533206507413


### Висновок до завдання 5

Було перевірено власну гіпотезу про розподіл категорій температурних відхилень:

- «холодніше» — 25%;
- «звичайно» — 25%;
- «тепліше» — 50%.

За допомогою критерію χ² порівняно фактичні та очікувані частоти.

Якщо p-value > 0.05, нульову гіпотезу не відхиляємо: фактичний розподіл не має статистично значущої відмінності від заданих пропорцій.

Якщо p-value < 0.05, нульову гіпотезу відхиляємо: фактичний розподіл статистично значуще відрізняється від заданих пропорцій.

Отже, рішення приймається на основі отриманого значення p-value при рівні значущості α = 0.05.

# Контрольні питання

### 1. Чому в статистиці χ² використовується квадрат відхилення і ділення на очікувану частоту?

Статистика χ² порівнює фактичні та очікувані частоти:

χ² = Σ (O - E)² / E

де O — спостережувана частота, а E — очікувана частота.

Квадрат відхилення потрібен для того, щоб додатні та від'ємні відхилення не взаємно компенсувалися. Ділення на очікувану частоту нормалізує величину відхилення: однакова абсолютна різниця має різну вагу для малих і великих очікуваних частот.

### 2. Як обчислюється очікувана частота клітинки таблиці спряженості?

Очікувана частота обчислюється за формулою:

E = (сума рядка × сума стовпця) / загальна сума.

Ця формула випливає з припущення незалежності змінних. Якщо змінні незалежні, частка певної категорії в одному вимірі не залежить від категорії іншого виміру. Тому добуток відповідних часток дозволяє отримати очікувану частку клітинки, а множення на загальну кількість спостережень дає очікувану частоту.

### 3. Що означає велике p-value?

Велике p-value означає, що немає достатніх статистичних підстав відхилити нульову гіпотезу H₀ при заданому рівні значущості.

Це НЕ означає, що H₀ доведена як істинна.

Наприклад, якщо p-value > 0.05, ми говоримо «немає підстав відхиляти H₀», а не «доведено, що змінні незалежні».

### 4. Що робити, якщо очікувана частота менша за 5?

Якщо очікувана частота в клітинці менша за 5, стандартне χ²-апроксимування може бути ненадійним.

Можливі рішення:

- об'єднати близькі або змістовно пов'язані категорії;
- збільшити обсяг вибірки;
- для невеликих таблиць використати відповідний точний критерій.

У цій роботі для відновлення умови застосовності було об'єднано категорії «звичайно» та «тепліше» у категорію «не холодніше», після чого критерій χ² було розраховано повторно.